## Data Acquisition: News API

In [1]:
import os
import re
import sys
import html

import pandas as pd

from datetime import datetime
from dotenv import load_dotenv
load_dotenv()

from newsapi import NewsApiClient

# Get the current working directory of the notebook
notebook_dir = os.getcwd()
# Add the parent directory to the system path
sys.path.append(os.path.join(notebook_dir, '../'))

from data_processing import DataProcessing
from feature_extraction import SpacyFeatureExtraction

In [2]:
pd.set_option('max_colwidth', 800)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)

In [3]:
api_key = os.getenv('NEWS_API_KEY')

In [4]:
QUERY_CONFIGS = [
    {"query_domain": "finance", "query": "(earnings OR revenue OR inflation OR interest rates) AND (expected OR forecast OR outlook OR projected)"},
    {"query_domain": "health", "query": "(disease OR virus OR vaccine OR public health) AND (expected OR forecast OR projected OR prognosis)"},
    {"query_domain": "weather", "query": "(weather OR storm OR hurricane OR rainfall OR temperature) AND (forecast OR expected OR outlook)"},
    {"query_domain": "policy", "query": "(election OR vote OR polling OR legislation) AND (expected OR likely OR projected OR forecast)"},
    {"query_domain": "sports_nfl", "query": "NFL AND (playoffs OR draft OR season) AND (expected OR likely OR prediction OR forecast)"},
    {"query_domain": "sports_nba", "query": "NBA AND (playoffs OR finals OR season) AND (expected OR likely OR prediction OR forecast)"},
    {"query_domain": "sports_college", "query": "(NCAA OR college football OR March Madness) AND (expected OR likely OR prediction OR forecast)"},
]

list_num = 1

In [5]:
# query_string = "Tesla OR SAE Level 4 autonomy"
start_date = "2026-03-21"
end_date = "2026-03-28"
newsapi = NewsApiClient(api_key=api_key)
all_articles = newsapi.get_everything(
    q=QUERY_CONFIGS[list_num]['query'],
    language="en",
    from_param=start_date,
    to=end_date,
    sort_by="relevancy",
    page_size=100
)

In [6]:
all_articles.keys()

dict_keys(['status', 'totalResults', 'articles'])

In [7]:
len(all_articles['articles']), all_articles['articles']

(94,
 [{'source': {'id': None, 'name': 'Gizmodo.com'},
   'author': 'Ed Cara',
   'title': 'Finally, Some Good News in the Fight Against Lyme Disease',
   'description': 'Scientists found that the experimental vaccine was more than 70% effective at preventing Lyme disease in a trial of nearly 10,000 volunteers.',
   'url': 'https://gizmodo.com/finally-some-good-news-in-the-fight-against-lyme-disease-2000737417',
   'urlToImage': 'https://gizmodo.com/app/uploads/2021/11/f2d874f6239357ae44984bcae121ed56.jpg',
   'publishedAt': '2026-03-24T19:35:35Z',
   'content': 'Ticks and the many diseases they can spread to people are a growing public health threat. With any luck, though, humanity may soon have a new weapon to use against the dreaded bloodsucking menace. A … [+4580 chars]'},
  {'source': {'id': None, 'name': 'nj.com'},
   'author': 'Steve Strunsky',
   'title': 'Closed Catholic school getting a new purpose in crowded section of N.J.’s largest city',
   'description': 'Details of the 

In [8]:
df = pd.DataFrame(all_articles['articles'])
df

,source,author,title,description,url,urlToImage,publishedAt,content
0,"{'id': None, 'name': 'Gizmodo.com'}",Ed Cara,"Finally, Some Good News in the Fight Against Lyme Disease","Scientists found that the experimental vaccine was more than 70% effective at preventing Lyme disease in a trial of nearly 10,000 volunteers.",https://gizmodo.com/finally-some-good-news-in-the-fight-against-lyme-disease-2000737417,https://gizmodo.com/app/uploads/2021/11/f2d874f6239357ae44984bcae121ed56.jpg,2026-03-24T19:35:35Z,"Ticks and the many diseases they can spread to people are a growing public health threat. With any luck, though, humanity may soon have a new weapon to use against the dreaded bloodsucking menace. A … [+4580 chars]"
1,"{'id': None, 'name': 'nj.com'}",Steve Strunsky,Closed Catholic school getting a new purpose in crowded section of N.J.’s largest city,"Details of the school’s projected opening date, grade configuration, cost and other aspects. would be made public during a special meeting at 5 p.m. on March...",https://www.nj.com/essex/2026/03/newark-plans-new-school-in-crowded-ironbound-section-on-site-of-shuttered-catholic-school.html,https://s.yimg.com/ny/api/res/1.2/ixkGljWVL3jaFAYti6DobA--/YXBwaWQ9aGlnaGxhbmRlcjt3PTEyMDA7aD05MDA7Y2Y9d2VicA--/https://media.zenfs.com/en/nj_com_articles_950/4d78b16399dd7696152f78e2f4b9c112,2026-03-22T15:00:46Z,"The Newark Board of Education president said the district plans to open a new school on the site of a shuttered former parochial school in the citys Ironbound section, using a lease-purchase agreemen… [+6313 chars]"
2,"{'id': None, 'name': 'Scientific American'}","Kendra Pierre-Louis, Andrea Thompson, Sushmita Pathak, Alex Sugiura","Spring heat dome, a blow to RFK, Jr.’s health agenda, SpaceX Starlink milestone","An unseasonal heat dome over parts of the U.S., a federal court ruling that blocks the CDC’s recent change to its recommended childhood vaccine schedule, new research on unsafe levels of lead in fast fashion",https://www.scientificamerican.com/podcast/episode/spring-heat-dome-a-blow-to-rfk-jr-s-health-agenda-spacex-starlink-milestone/,https://static.scientificamerican.com/dam/m/3557c42768a73913/original/2603_SQ_MON_MARCH_23.png?m=1774038209.608&w=1200,2026-03-23T10:00:00Z,"Kendra Pierre-Louis: For Scientific American’s Science Quickly, I’m Kendra Pierre-Louis, in for Rachel Feltman. You’re listening to our weekly science news roundup.\r\nIf you live in much of California… [+10640 chars]"
3,"{'id': None, 'name': 'NPR'}",Aaron Bolton,A $50 billion fund to help rural hospitals could actually lead to closures and cuts,"States are rolling out plans for their share of a $50 billion fund established by Congress to improve rural health care. In some states, the money may provoke rural hospitals to cut services.",https://www.npr.org/2026/03/26/nx-s1-5760925/rural-health-transformation-fund-hospitals-montana,https://npr.brightspotcdn.com/dims3/default/strip/false/crop/4032x2268+0+378/resize/1400/quality/85/format/jpeg/?url=http%3A%2F%2Fnpr-brightspot.s3.amazonaws.com%2Fc3%2Ffe%2F0b795951485a84cd8ec7ee284ea2%2Frht-03.jpg,2026-03-26T09:00:00Z,"The emergency department at Big Sandy Medical Center in Montana is just one room, with a single curtain between two beds.\r\nIt's one of the many parts of the 25-bed rural hospital that need updating, … [+8493 chars]"
4,"{'id': None, 'name': 'Gizmodo.com'}",James Pero,Oura’s Next Smart Ring Just Leaked a Year Early,"If these images are real, this is the first look at the Oura Ring 5.",https://gizmodo.com/ouras-next-smart-ring-just-leaked-a-year-early-2000737357,https://gizmodo.com/app/uploads/2025/11/Oura-Ring-4-Ceramic-review-05-1200x675.jpg,2026-03-24T15:40:39Z,"The Oura Ring 5 isn’t expected to launch for another year, but a purported leak might have given us an early glimpse of what to expect. Android Headlines claims to have obtained renders of the new he… [+2151 chars]"
...,...,...,...,...,...,...,...,...
89,"{'id': None, 'name': 'Nesbitt.io'}",A

In [9]:
df = df.rename(columns={
    "source": "Source Meta Data",
    "author": "Source",
    "publishedAt": "Date",
    "url": "URL",
    "urlToImage": "Image URL"
})

df["Query Domain"] = QUERY_CONFIGS[list_num]["query_domain"]

df

,Source Meta Data,Source,title,description,URL,Image URL,Date,content,Query Domain
0,"{'id': None, 'name': 'Gizmodo.com'}",Ed Cara,"Finally, Some Good News in the Fight Against Lyme Disease","Scientists found that the experimental vaccine was more than 70% effective at preventing Lyme disease in a trial of nearly 10,000 volunteers.",https://gizmodo.com/finally-some-good-news-in-the-fight-against-lyme-disease-2000737417,https://gizmodo.com/app/uploads/2021/11/f2d874f6239357ae44984bcae121ed56.jpg,2026-03-24T19:35:35Z,"Ticks and the many diseases they can spread to people are a growing public health threat. With any luck, though, humanity may soon have a new weapon to use against the dreaded bloodsucking menace. A … [+4580 chars]",health
1,"{'id': None, 'name': 'nj.com'}",Steve Strunsky,Closed Catholic school getting a new purpose in crowded section of N.J.’s largest city,"Details of the school’s projected opening date, grade configuration, cost and other aspects. would be made public during a special meeting at 5 p.m. on March...",https://www.nj.com/essex/2026/03/newark-plans-new-school-in-crowded-ironbound-section-on-site-of-shuttered-catholic-school.html,https://s.yimg.com/ny/api/res/1.2/ixkGljWVL3jaFAYti6DobA--/YXBwaWQ9aGlnaGxhbmRlcjt3PTEyMDA7aD05MDA7Y2Y9d2VicA--/https://media.zenfs.com/en/nj_com_articles_950/4d78b16399dd7696152f78e2f4b9c112,2026-03-22T15:00:46Z,"The Newark Board of Education president said the district plans to open a new school on the site of a shuttered former parochial school in the citys Ironbound section, using a lease-purchase agreemen… [+6313 chars]",health
2,"{'id': None, 'name': 'Scientific American'}","Kendra Pierre-Louis, Andrea Thompson, Sushmita Pathak, Alex Sugiura","Spring heat dome, a blow to RFK, Jr.’s health agenda, SpaceX Starlink milestone","An unseasonal heat dome over parts of the U.S., a federal court ruling that blocks the CDC’s recent change to its recommended childhood vaccine schedule, new research on unsafe levels of lead in fast fashion",https://www.scientificamerican.com/podcast/episode/spring-heat-dome-a-blow-to-rfk-jr-s-health-agenda-spacex-starlink-milestone/,https://static.scientificamerican.com/dam/m/3557c42768a73913/original/2603_SQ_MON_MARCH_23.png?m=1774038209.608&w=1200,2026-03-23T10:00:00Z,"Kendra Pierre-Louis: For Scientific American’s Science Quickly, I’m Kendra Pierre-Louis, in for Rachel Feltman. You’re listening to our weekly science news roundup.\r\nIf you live in much of California… [+10640 chars]",health
3,"{'id': None, 'name': 'NPR'}",Aaron Bolton,A $50 billion fund to help rural hospitals could actually lead to closures and cuts,"States are rolling out plans for their share of a $50 billion fund established by Congress to improve rural health care. In some states, the money may provoke rural hospitals to cut services.",https://www.npr.org/2026/03/26/nx-s1-5760925/rural-health-transformation-fund-hospitals-montana,https://npr.brightspotcdn.com/dims3/default/strip/false/crop/4032x2268+0+378/resize/1400/quality/85/format/jpeg/?url=http%3A%2F%2Fnpr-brightspot.s3.amazonaws.com%2Fc3%2Ffe%2F0b795951485a84cd8ec7ee284ea2%2Frht-03.jpg,2026-03-26T09:00:00Z,"The emergency department at Big Sandy Medical Center in Montana is just one room, with a single curtain between two beds.\r\nIt's one of the many parts of the 25-bed rural hospital that need updating, … [+8493 chars]",health
4,"{'id': None, 'name': 'Gizmodo.com'}",James Pero,Oura’s Next Smart Ring Just Leaked a Year Early,"If these images are real, this is the first look at the Oura Ring 5.",https://gizmodo.com/ouras-next-smart-ring-just-leaked-a-year-early-2000737357,https://gizmodo.com/app/uploads/2025/11/Oura-Ring-4-Ceramic-review-05-1200x675.jpg,2026-03-24T15:40:39Z,"The Oura Ring 5 isn’t expected to launch for another year, but a purported leak might have given us an early glimpse of what to expect. Android Headlines claims to have obtained renders of the new he… [+2151 chars]",health
...,...,...,...,...,...,...

In [10]:
def clean_base_sentence(text: str) -> str:
    if not isinstance(text, str):
        return text

    # Decode HTML entities (e.g., &lt;li&gt;)
    text = html.unescape(text)

    # Remove UL/LI tags but keep structure
    text = re.sub(r"</?ul>", "", text)
    text = re.sub(r"<li>", "", text)
    text = re.sub(r"</li>", " ", text)

    # Normalize newlines and carriage returns → space
    text = re.sub(r"[\r\n]+", " ", text)

    # Collapse multiple spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [11]:
def build_prediction_dataframe(df, char_limit=500):
    """
    Builds a sentence-level dataframe from a preprocessed NewsAPI DataFrame.
    """

    df = df.reset_index(drop=True).copy()
    df["article_id"] = df.index

    prediction_keywords = [
        "expected", "forecast", "forecasted", "forecasting",
        "forecast estimate", "forecasted outcome", "projected",
        "projection", "predict", "predicted", "estimate",
        "expectation", "anticipation", "prognosis",
        "speculation", "foretelling", "prophecy", "guess"
    ]

    pattern = r"\b(" + "|".join(map(re.escape, prediction_keywords)) + r")\b"

    df["prediction_visible"] = (
        (
            df["title"].fillna("") +
            df["description"].fillna("") +
            df["content"].fillna("")
        )
        .str.lower()
        .str[:char_limit]
        .str.contains(pattern, regex=True)
    ).astype(int)

    parts = []
    for field, order in [("title", 0), ("description", 1), ("content", 2)]:
        tmp = df.copy()
        tmp["Base Sentence"] = tmp[field]
        tmp["Source_Field"] = field
        tmp["field_order"] = order
        parts.append(tmp)

    long_df = pd.concat(parts, ignore_index=True)

    # ✅ CLEAN TEXT FIRST
    long_df["Base Sentence"] = long_df["Base Sentence"].apply(clean_base_sentence)

    # ✅ SPLIT INTO SENTENCES HERE (USING self.nlp)
    sfe = SpacyFeatureExtraction(long_df, "Base Sentence")
    long_df = sfe.split_into_sentences()

    # ✅ LABEL PER SENTENCE (NOT PER BLOCK)
    long_df["Sentence Label"] = (
        long_df["Base Sentence"]
        .str.lower()
        .str.contains(pattern, regex=True)
        .astype(int)
    )

    long_df["Human Annotation"] = ""
    long_df["Human Reasoning"] = ""

    priority_cols = [
        "Base Sentence", "Sentence Label",
        "Human Annotation", "Human Reasoning",
        "Source", "Date"
    ]
    priority_cols = [c for c in priority_cols if c in long_df.columns]
    remaining_cols = [c for c in long_df.columns if c not in priority_cols]

    return long_df[priority_cols + remaining_cols]

In [12]:
final_df = build_prediction_dataframe(df)
final_df.head(3)

/var/folders/78/9z0b45fx1xqbwxh8vk97lcfh0000gn/T/ipykernel_56648/1523799737.py:27: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(pattern, regex=True)
/var/folders/78/9z0b45fx1xqbwxh8vk97lcfh0000gn/T/ipykernel_56648/1523799737.py:51: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(pattern, regex=True)


,Base Sentence,Sentence Label,Human Annotation,Human Reasoning,Source,Date,Source Meta Data,title,description,URL,Image URL,content,Query Domain,article_id,prediction_visible,Source_Field,field_order
0,"Finally, Some Good News in the Fight Against Lyme Disease",0,,,Ed Cara,2026-03-24T19:35:35Z,"{'id': None, 'name': 'Gizmodo.com'}","Finally, Some Good News in the Fight Against Lyme Disease","Scientists found that the experimental vaccine was more than 70% effective at preventing Lyme disease in a trial of nearly 10,000 volunteers.",https://gizmodo.com/finally-some-good-news-in-the-fight-against-lyme-disease-2000737417,https://gizmodo.com/app/uploads/2021/11/f2d874f6239357ae44984bcae121ed56.jpg,"Ticks and the many diseases they can spread to people are a growing public health threat. With any luck, though, humanity may soon have a new weapon to use against the dreaded bloodsucking menace. A … [+4580 chars]",health,0,0,title,0
1,Closed Catholic school getting a new purpose in crowded section of N.J.’s largest city,0,,,Steve Strunsky,2026-03-22T15:00:46Z,"{'id': None, 'name': 'nj.com'}",Closed Catholic school getting a new purpose in crowded section of N.J.’s largest city,"Details of the school’s projected opening date, grade configuration, cost and other aspects. would be made public during a special meeting at 5 p.m. on March...",https://www.nj.com/essex/2026/03/newark-plans-new-school-in-crowded-ironbound-section-on-site-of-shuttered-catholic-school.html,https://s.yimg.com/ny/api/res/1.2/ixkGljWVL3jaFAYti6DobA--/YXBwaWQ9aGlnaGxhbmRlcjt3PTEyMDA7aD05MDA7Y2Y9d2VicA--/https://media.zenfs.com/en/nj_com_articles_950/4d78b16399dd7696152f78e2f4b9c112,"The Newark Board of Education president said the district plans to open a new school on the site of a shuttered former parochial school in the citys Ironbound section, using a lease-purchase agreemen… [+6313 chars]",health,1,1,title,0
2,"Spring heat dome, a blow to RFK, Jr.’s health agenda, SpaceX Starlink milestone",0,,,"Kendra Pierre-Louis, Andrea Thompson, Sushmita Pathak, Alex Sugiura",2026-03-23T10:00:00Z,"{'id': None, 'name': 'Scientific American'}","Spring heat dome, a blow to RFK, Jr.’s health agenda, SpaceX Starlink milestone","An unseasonal heat dome over parts of the U.S., a federal court ruling that blocks the CDC’s recent change to its recommended childhood vaccine schedule, new research on unsafe levels of lead in fast fashion",https://www.scientificamerican.com/podcast/episode/spring-heat-dome-a-blow-to-rfk-jr-s-health-agenda-spacex-starlink-milestone/,https://static.scientificamerican.com/dam/m/3557c42768a73913/original/2603_SQ_MON_MARCH_23.png?m=1774038209.608&w=1200,"Kendra Pierre-Louis: For Scientific American’s Science Quickly, I’m Kendra Pierre-Louis, in for Rachel Feltman. You’re listening to our weekly science news roundup.\r\nIf you live in much of California… [+10640 chars]",health,2,0,title,0


In [13]:
predictions_df = final_df[final_df["Sentence Label"] == 1]
predictions_df.head(3)

,Base Sentence,Sentence Label,Human Annotation,Human Reasoning,Source,Date,Source Meta Data,title,description,URL,Image URL,content,Query Domain,article_id,prediction_visible,Source_Field,field_order
21,Trump is expected to name a new CDC head.,1,,,Erika Edwards,2026-03-24T21:39:26Z,"{'id': 'nbc-news', 'name': 'NBC News'}","Trump is expected to name a new CDC head. ‘We don’t need a TV personality,"" one expert said.","As the Trump administration prepares to nominate a new director of the CDC, insiders say they worry the nominee will only further undermine trust in the nation’s top health agency.",https://www.nbcnews.com/health/health-news/trump-cdc-director-nominate-vaccination-measles-rcna261938,"https://media-cldnry.s-nbcnews.com/image/upload/t_nbcnews-fp-1200-630,f_auto,q_auto:best/rockcms/2025-12/251201-cdc-mn-1300-5a5092.jpg","As the Trump administration prepares to nominate a new director of the Centers for Disease Control and Prevention, insiders say they worry the nominee will only further undermine trust in the nations… [+6153 chars]",health,19,1,title,0
81,"Employee Is Turned Down For The Role Of Supervisor, But She’s Still Expected To Do Tasks A Supervisor Should Do",1,,,Jayne Elliott,2026-03-24T18:55:05Z,"{'id': None, 'name': 'Twistedsifter.com'}","Employee Is Turned Down For The Role Of Supervisor, But She’s Still Expected To Do Tasks A Supervisor Should Do",Jaded employees won't go above and beyond.,http://twistedsifter.com/2026/03/employee-is-turned-down-for-the-role-of-supervisor-but-shes-still-expected-to-do-tasks-a-supervisor-should-do/,https://twistedsifter.com/wp-content/uploads/2026/02/Twisted-Sifter-17.png,"Shutterstock/Reddit\r\nImagine working for a company for awhile and hoping to get promoted to supervisor. If you were turned down for the promotion, would you be willing to take on extra tasks to be a … [+2785 chars]",health,72,1,title,0
93,"Mets projected lineup, rotation and keys to winning NL East",1,,,"Andrew Tredinnick, NorthJersey.com",2026-03-24T08:04:55Z,"{'id': None, 'name': 'NorthJersey.com'}","Mets projected lineup, rotation and keys to winning NL East",The Mets begin their 2026 'redemption tour' on Opening Day after a disappointing 2025 season. See the expected new lineup and rotation.,https://www.northjersey.com/story/sports/mlb/mets/2026/03/24/ny-mets-2026-preview-opening-day-lineup-rotation-predictions/89249791007/,https://s.yimg.com/ny/api/res/1.2/zsxORNdWfVeSB.BDPAT_0Q--/YXBwaWQ9aGlnaGxhbmRlcjt3PTEyMDA7aD04MDA7Y2Y9d2VicA--/https://media.zenfs.com/en/the-bergen-record/cd3358ba16b0cc83e81a48f238e60174,"The Mets' redemption tour begins on Thursday afternoon after a major bust in the 2025 season.\r\nAfter falling one win short of a playoff spot, Opening Day marks the start of a new era for the Mets aft… [+8944 chars]",health,82,1,title,0


In [14]:
predictions_df["Base Sentence"] = predictions_df["Base Sentence"].apply(clean_base_sentence)
predictions_df.head(3)

,Base Sentence,Sentence Label,Human Annotation,Human Reasoning,Source,Date,Source Meta Data,title,description,URL,Image URL,content,Query Domain,article_id,prediction_visible,Source_Field,field_order
21,Trump is expected to name a new CDC head.,1,,,Erika Edwards,2026-03-24T21:39:26Z,"{'id': 'nbc-news', 'name': 'NBC News'}","Trump is expected to name a new CDC head. ‘We don’t need a TV personality,"" one expert said.","As the Trump administration prepares to nominate a new director of the CDC, insiders say they worry the nominee will only further undermine trust in the nation’s top health agency.",https://www.nbcnews.com/health/health-news/trump-cdc-director-nominate-vaccination-measles-rcna261938,"https://media-cldnry.s-nbcnews.com/image/upload/t_nbcnews-fp-1200-630,f_auto,q_auto:best/rockcms/2025-12/251201-cdc-mn-1300-5a5092.jpg","As the Trump administration prepares to nominate a new director of the Centers for Disease Control and Prevention, insiders say they worry the nominee will only further undermine trust in the nations… [+6153 chars]",health,19,1,title,0
81,"Employee Is Turned Down For The Role Of Supervisor, But She’s Still Expected To Do Tasks A Supervisor Should Do",1,,,Jayne Elliott,2026-03-24T18:55:05Z,"{'id': None, 'name': 'Twistedsifter.com'}","Employee Is Turned Down For The Role Of Supervisor, But She’s Still Expected To Do Tasks A Supervisor Should Do",Jaded employees won't go above and beyond.,http://twistedsifter.com/2026/03/employee-is-turned-down-for-the-role-of-supervisor-but-shes-still-expected-to-do-tasks-a-supervisor-should-do/,https://twistedsifter.com/wp-content/uploads/2026/02/Twisted-Sifter-17.png,"Shutterstock/Reddit\r\nImagine working for a company for awhile and hoping to get promoted to supervisor. If you were turned down for the promotion, would you be willing to take on extra tasks to be a … [+2785 chars]",health,72,1,title,0
93,"Mets projected lineup, rotation and keys to winning NL East",1,,,"Andrew Tredinnick, NorthJersey.com",2026-03-24T08:04:55Z,"{'id': None, 'name': 'NorthJersey.com'}","Mets projected lineup, rotation and keys to winning NL East",The Mets begin their 2026 'redemption tour' on Opening Day after a disappointing 2025 season. See the expected new lineup and rotation.,https://www.northjersey.com/story/sports/mlb/mets/2026/03/24/ny-mets-2026-preview-opening-day-lineup-rotation-predictions/89249791007/,https://s.yimg.com/ny/api/res/1.2/zsxORNdWfVeSB.BDPAT_0Q--/YXBwaWQ9aGlnaGxhbmRlcjt3PTEyMDA7aD04MDA7Y2Y9d2VicA--/https://media.zenfs.com/en/the-bergen-record/cd3358ba16b0cc83e81a48f238e60174,"The Mets' redemption tour begins on Thursday afternoon after a major bust in the 2025 season.\r\nAfter falling one win short of a playoff spot, Opening Day marks the start of a new era for the Mets aft… [+8944 chars]",health,82,1,title,0


In [15]:
predictions_df

,Base Sentence,Sentence Label,Human Annotation,Human Reasoning,Source,Date,Source Meta Data,title,description,URL,Image URL,content,Query Domain,article_id,prediction_visible,Source_Field,field_order
21,Trump is expected to name a new CDC head.,1,,,Erika Edwards,2026-03-24T21:39:26Z,"{'id': 'nbc-news', 'name': 'NBC News'}","Trump is expected to name a new CDC head. ‘We don’t need a TV personality,"" one expert said.","As the Trump administration prepares to nominate a new director of the CDC, insiders say they worry the nominee will only further undermine trust in the nation’s top health agency.",https://www.nbcnews.com/health/health-news/trump-cdc-director-nominate-vaccination-measles-rcna261938,"https://media-cldnry.s-nbcnews.com/image/upload/t_nbcnews-fp-1200-630,f_auto,q_auto:best/rockcms/2025-12/251201-cdc-mn-1300-5a5092.jpg","As the Trump administration prepares to nominate a new director of the Centers for Disease Control and Prevention, insiders say they worry the nominee will only further undermine trust in the nations… [+6153 chars]",health,19,1,title,0
81,"Employee Is Turned Down For The Role Of Supervisor, But She’s Still Expected To Do Tasks A Supervisor Should Do",1,,,Jayne Elliott,2026-03-24T18:55:05Z,"{'id': None, 'name': 'Twistedsifter.com'}","Employee Is Turned Down For The Role Of Supervisor, But She’s Still Expected To Do Tasks A Supervisor Should Do",Jaded employees won't go above and beyond.,http://twistedsifter.com/2026/03/employee-is-turned-down-for-the-role-of-supervisor-but-shes-still-expected-to-do-tasks-a-supervisor-should-do/,https://twistedsifter.com/wp-content/uploads/2026/02/Twisted-Sifter-17.png,"Shutterstock/Reddit\r\nImagine working for a company for awhile and hoping to get promoted to supervisor. If you were turned down for the promotion, would you be willing to take on extra tasks to be a … [+2785 chars]",health,72,1,title,0
93,"Mets projected lineup, rotation and keys to winning NL East",1,,,"Andrew Tredinnick, NorthJersey.com",2026-03-24T08:04:55Z,"{'id': None, 'name': 'NorthJersey.com'}","Mets projected lineup, rotation and keys to winning NL East",The Mets begin their 2026 'redemption tour' on Opening Day after a disappointing 2025 season. See the expected new lineup and rotation.,https://www.northjersey.com/story/sports/mlb/mets/2026/03/24/ny-mets-2026-preview-opening-day-lineup-rotation-predictions/89249791007/,https://s.yimg.com/ny/api/res/1.2/zsxORNdWfVeSB.BDPAT_0Q--/YXBwaWQ9aGlnaGxhbmRlcjt3PTEyMDA7aD04MDA7Y2Y9d2VicA--/https://media.zenfs.com/en/the-bergen-record/cd3358ba16b0cc83e81a48f238e60174,"The Mets' redemption tour begins on Thursday afternoon after a major bust in the 2025 season.\r\nAfter falling one win short of a playoff spot, Opening Day marks the start of a new era for the Mets aft… [+8944 chars]",health,82,1,title,0
108,"Details of the school’s projected opening date, grade configuration, cost and other aspects.",1,,,Steve Strunsky,2026-03-22T15:00:46Z,"{'id': None, 'name': 'nj.com'}",Closed Catholic school getting a new purpose in crowded section of N.J.’s largest city,"Details of the school’s projected opening date, grade configuration, cost and other aspects. would be made public during a special meeting at 5 p.m. on March...",https://www.nj.com/essex/2026/03/newark-plans-new-school-in-crowded-ironbound-section-on-site-of-shuttered-catholic-school.html,https://s.yimg.com/ny/api/res/1.2/ixkGljWVL3jaFAYti6DobA--/YXBwaWQ9aGlnaGxhbmRlcjt3PTEyMDA7aD05MDA7Y2Y9d2VicA--/https://media.zenfs.com/en/nj_com_articles_950/4d78b16399dd7696152f78e2f4b9c112,"The Newark Board of Education president said the district plans to open a new school on the site of a shuttered former parochial school in the citys Ironbound section, using a lease-purchase agreemen… [+6313 chars]",health,1,1,description,1
177,"The administration is expected to soon name a new director, who will have their hands full.",1,,,Pien Huang,2026-03-25T16:01:04Z,"{'id': None, 'name': 'NPR'}",A leadership vacuum adds t

In [16]:
len(predictions_df)

21

In [ ]:
def build_query_package(
    *,
    query_string: str,
    query_domain: str, 
    start_date: str,
    end_date: str,
    df,
    final_df,
    predictions_df,
    language: str = "en",
    sort_by: str = "relevancy",
    page_size: int = None
):
    """
    Builds a query-aligned package and a deterministic filename prefix.
    """

    # -------------------------
    # Prefix from query
    # -------------------------
    base_prefix = (
        query_string.lower()
        .replace(" or ", "_")
        .replace(" and ", "_")
        .replace(" ", "_")
    )
    base_prefix = re.sub(r"[^a-z0-9_]", "", base_prefix)
    base_prefix = re.sub(r"_+", "_", base_prefix).strip("_")

    prefix = f"{base_prefix}_{start_date}_to_{end_date}"

    # -------------------------
    # Query metadata
    # -------------------------
    query_meta = {
        "query": query_string,
        "query_domain": query_domain,
        "language": language,
        "from_date": start_date,
        "to_date": end_date,
        "sort_by": sort_by,
        "page_size": page_size,
        "timestamp_utc": datetime.utcnow().isoformat()
    }

    # -------------------------
    # Full package
    # -------------------------
    package = {
        "query_meta": query_meta,
        "counts": {
            "num_raw_articles": len(df),
            "num_sentences": len(final_df),
            "num_potential_predictions": len(predictions_df)
        },
        "raw_articles": df.to_dict(orient="records"),
        "processed_sentences": final_df.to_dict(orient="records"),
        "potential_predictions": predictions_df.to_dict(orient="records")
    }

    return prefix, package

In [ ]:
prefix, query_package = build_query_package(
    query_string=QUERY_CONFIGS[list_num]["query"],
    query_domain=QUERY_CONFIGS[list_num]["query_domain"],
    start_date=start_date,
    end_date=end_date,
    df=df,
    final_df=final_df,
    predictions_df=predictions_df,
    page_size=7
)

In [ ]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir)
save_path = os.path.join(base_data_path, 'news_api', 'annotators')

DataProcessing.save_to_file(
    data=predictions_df,
    path=save_path,
    prefix=f"{prefix}_predictions",
    save_file_type="csv"
)

In [ ]:
DataProcessing.save_to_file(
    data=query_package,
    path=save_path,
    prefix=f"{prefix}_full",
    save_file_type="json"
)